# 第11章 シーンの作成と編集

書籍『Sionna RTによる電波伝搬レイトレーシング入門』第11章のコードである。
セルは本文の掲載順に並べてあり、コードセルの中身は本文に印刷したものと同じである。

先頭の「準備」セルだけは本文に印刷していない。本文のコードが前提にしている
import とシーンの読み込みをここで済ませてある。準備セルを実行すれば、以降は
上から順に実行できる。

ただし、本文が説明のために示している断片（自分で作ったシーンのパスや、その
シーンには無い物体名を使う例）はそのままでは動かない。該当するセルの前に注記を
置いてある。

## 準備

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sionna.rt as rt
from sionna.rt import (load_scene, scene as scene_assets,
                       Transmitter, Receiver, PlanarArray,
                       PathSolver, RadioMapSolver, Camera, RadioMaterial)

# 本書の多くの章で使うミュンヘン中心部の都市シーン
scene = load_scene(scene_assets.munich)
scene.frequency = 3.5e9
scene.tx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.rx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.add(Transmitter('tx0', position=[-30.0, 38.0, 35.0]))

solver = PathSolver()
rmsolver = RadioMapSolver()

## 1. Sionna RT のシーン構造

In [ ]:
print(scene.objects)
# {'building_001': <SceneObject ...>, 'building_002': ..., 'ground': ...}

```xml
<scene version="2.1.0">
  <bsdf type="itu-radio-material" id="wall-mat">
    <string name="type" value="concrete"/>
    <float name="thickness" value="0.2"/>
  </bsdf>
  <shape type="rectangle" id="wall">
    <transform name="to_world">
      <scale x="5" y="5" z="1"/>
      <translate x="0" y="0" z="0"/>
    </transform>
    <ref id="wall-mat"/>
  </shape>
</scene>
```

## 3. シーンを読み込む

**注記**: 自分で作ったシーンを読む例である。パスを自分の `scene.xml` に置き換えて実行する。

In [ ]:
from sionna.rt import load_scene, scene as scene_assets

scene = load_scene('scenes/my_scene/scene.xml')  # 自作シーン
builtin = load_scene(scene_assets.munich)        # 組み込みシーン

## 4. 物体を追加する

In [ ]:
from sionna.rt import SceneObject, ITURadioMaterial
from sionna.rt import scene as scene_assets

metal = ITURadioMaterial('obstacle-material', 'metal',
                         thickness=0.01)
obstacle = SceneObject(fname=scene_assets.low_poly_car,
                       name='obstacle', radio_material=metal)
scene.edit(add=obstacle)       # 先にシーンへ入れる
obstacle.position = [0, 0, 0]  # 位置は追加した後で決める

## 5. 物体を削除する

In [ ]:
scene.edit(remove=obstacle)

## 6. 物体を移動する

In [ ]:
obj = scene.objects['building_001']
obj.position = [obj.position[0]+10, obj.position[1], obj.position[2]]

## 7. 物体を回転する

In [ ]:
import numpy as np
obj.orientation = [np.pi/2, 0, 0]  # z 軸まわりに 90 度

## 8. 材料を変更する

In [ ]:
from sionna.rt import RadioMaterial
metal = RadioMaterial(name='metal', relative_permittivity=1.0,
                      conductivity=1e7)
obj.radio_material = metal